# 02 - Feature Engineering, Benchmarking & OOD Evaluation

In Notebook 1, Random Forest, XGBoost, and LightGBM all plateaued around 86% accuracy (~0.875 Macro F1) on 36 features, with ~80% of errors falling into `Normal` and `DDoS_HTTP`. Here we extract payload/URI character lengths and destination port tiers before dropping raw strings and ports, remove `r = 1.00` collinear MQTT columns, re-benchmark all three models, and stress-test the final XGBoost pipeline on unseen and out-of-distribution flows.

In [ ]:
!git clone -q https://github.com/PriyoChand/smart-campus-intrusion-detection.git
%cd smart-campus-intrusion-detection

!pip install -q dvc lightgbm xgboost pyarrow joblib

In [ ]:
import getpass
from google.colab import userdata

DAGSHUB_USER = "PriyoChand"
try:
    DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")
except Exception:
    DAGSHUB_TOKEN = getpass.getpass("DAGsHub Token: ")

!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {DAGSHUB_USER}
!dvc remote modify origin --local password {DAGSHUB_TOKEN}
!dvc pull -q -r origin

## 1. Feature Engineering & Cleaning

Before dropping raw text and port columns, we compute:
- Character lengths (`*_len`) for 6 payload/URI fields (mapping inactive placeholders to `0`).
- Destination port buckets (`tcp_dstport_tier`, `udp_port_tier`): `0` inactive, `1` well-known (`<1024`), `2` registered (`1024-49151`), `3` ephemeral (`>=49152`).

We then drop identity/counter leakage columns, the raw text/port columns, the 8 zero-variance columns identified in Notebook 1, and 4 exact MQTT duplicates (`r = 1.00`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_PATH = "data/raw/ML-EdgeIIoT-dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw shape: {df.shape}")

inactive_tokens = ["0", "0.0", "-", ";", "?", "None", "nan", ""]

def extract_length(series):
    s = series.astype(str).str.strip()
    return np.where(s.isin(inactive_tokens), 0, s.str.len()).astype(np.float32)

def bucket_port(series):
    p = pd.to_numeric(series, errors="coerce").fillna(0)
    return np.select(
        [p <= 0, p < 1024, p < 49152, p >= 49152],
        [0, 1, 2, 3],
        default=0
    ).astype(np.int8)

raw_text_cols = [
    "http.file_data", "http.request.full_uri", "http.request.uri.query",
    "tcp.options", "tcp.payload", "mqtt.msg"
]
for col in raw_text_cols:
    if col in df.columns:
        df[col.replace(".", "_") + "_len"] = extract_length(df[col])

for col in ["tcp.dstport", "udp.port"]:
    if col in df.columns:
        df[col.replace(".", "_") + "_tier"] = bucket_port(df[col])

drop_cols = [
    # Leakage & capture counters
    "frame.time", "ip.src_host", "ip.dst_host",
    "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
    "icmp.transmit_timestamp", "udp.stream", "tcp.seq",
    # Raw strings & ports (summarized above)
    "http.file_data", "http.request.full_uri", "http.request.uri.query",
    "tcp.options", "tcp.payload", "mqtt.msg",
    "tcp.srcport", "tcp.dstport", "udp.port",
    # Zero-variance columns
    "icmp.unused", "http.tls_port", "dns.qry.type", "dns.retransmit_request_in",
    "mqtt.msg_decoded_as", "mbtcp.len", "mbtcp.trans_id", "mbtcp.unit_id",
    # Exact collinear clones (r = 1.00)
    "mqtt.msgtype", "mqtt.conflag.cleansess", "mqtt.conflags", "mqtt.proto_len"
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

target_cols = ["Attack_label", "Attack_type"]
cat_cols = [
    c for c in [
        "http.request.method", "http.referer", "http.request.version",
        "dns.qry.name.len", "mqtt.conack.flags", "mqtt.protoname", "mqtt.topic"
    ] if c in df.columns
]
num_cols = [c for c in df.columns if c not in cat_cols + target_cols]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().replace(inactive_tokens, "0")

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# Sanity check for any newly created zero-variance features
dead_cols = [c for c in num_cols + cat_cols if df[c].nunique() <= 1]
if dead_cols:
    print(f"Dropping zero-variance features: {dead_cols}")
    df.drop(columns=dead_cols, inplace=True)
    num_cols = [c for c in num_cols if c not in dead_cols]
    cat_cols = [c for c in cat_cols if c not in dead_cols]

n_before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dropped {n_before - len(df):,} exact duplicates -> Clean shape: {df.shape}")
print(f"Active features: {len(num_cols)} numerical, {len(cat_cols)} categorical ({len(num_cols) + len(cat_cols)} total)")

## 2. Stratified Split & Preprocessing Pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop(columns=target_cols)
le = LabelEncoder()
y = le.fit_transform(df["Attack_type"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="0")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)
preprocessor.set_output(transform="pandas")

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Train: {X_train_proc.shape} | Test: {X_test_proc.shape}")

## 3. Benchmarking Engineered Features (Random Forest vs. XGBoost vs. LightGBM)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Update the Random Forest baseline values after running Notebook 1
comparison = [
    {"Stage": "Baseline (36 feat)", "Model": "Random Forest", "Accuracy": 0.8384,   "Macro F1": 0.8498},
    {"Stage": "Baseline (36 feat)", "Model": "XGBoost",       "Accuracy": 0.8605, "Macro F1": 0.8753},
    {"Stage": "Baseline (36 feat)", "Model": "LightGBM",      "Accuracy": 0.8591, "Macro F1": 0.8727},
]

def run_eval(model_name, y_true, y_pred, show_report=False):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    comparison.append({
        "Stage": f"Engineered ({X_train_proc.shape[1]} feat)",
        "Model": model_name,
        "Accuracy": round(acc, 4),
        "Macro F1": round(macro_f1, 4),
    })
    print(f"{model_name} -> Acc: {acc:.4f} | Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f}")
    if show_report:
        print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

rf_eng = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1
)
rf_eng.fit(X_train_proc, y_train)
y_pred_rf = rf_eng.predict(X_test_proc)
run_eval("Random Forest", y_test, y_pred_rf)

xgb_eng = XGBClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1,
    random_state=42, n_jobs=-1, eval_metric="mlogloss"
)
xgb_eng.fit(X_train_proc, y_train)
y_pred_xgb = xgb_eng.predict(X_test_proc)
run_eval("XGBoost", y_test, y_pred_xgb, show_report=True)

lgb_eng = LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_eng.fit(X_train_proc, y_train)
y_pred_lgb = lgb_eng.predict(X_test_proc)
run_eval("LightGBM", y_test, y_pred_lgb)

display(pd.DataFrame(comparison))

In [ ]:
normal_id = le.transform(["Normal"])[0]
y_test_bin = (y_test != normal_id).astype(int)
y_pred_bin = (y_pred_xgb != normal_id).astype(int)
print(
    f"Derived Binary Detection (Normal vs Attack) -> "
    f"Accuracy: {accuracy_score(y_test_bin, y_pred_bin)*100:.2f}%, "
    f"F1: {f1_score(y_test_bin, y_pred_bin):.4f}"
)

feat_imp = pd.DataFrame({
    "Feature": X_train_proc.columns,
    "Importance (%)": (xgb_eng.feature_importances_ * 100).round(2)
}).sort_values("Importance (%)", ascending=False)

plt.figure(figsize=(11, 6))
sns.barplot(
    data=feat_imp.head(15), x="Importance (%)", y="Feature",
    hue="Feature", palette="mako", legend=False
)
plt.title(f"Top 15 Feature Importances (Engineered XGBoost — {X_train_proc.shape[1]} Features)", fontweight="bold")
plt.ylabel("")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 10))
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_
)
plt.title("15-Class Confusion Matrix (Engineered XGBoost)", fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Synthetic & Out-of-Distribution Stress Test

To check how the 40-feature XGBoost model behaves outside the holdout split, we evaluate 12 synthetic flows covering known attack families, malformed inputs (`NaN` numerics / unseen tokens), and non-Edge-IIoTset campus traffic. Predictions with confidence `< 50%` or Shannon entropy `> 2.5` bits are routed to an `Unknown Anomaly` bucket instead of forcing a class label.

In [ ]:
def make_flow(**overrides):
    row = {c: 0.0 for c in num_cols}
    row.update({c: "0" for c in cat_cols})
    for k, v in overrides.items():
        if k in row:
            row[k] = v
    return row

scenarios = [
    # In-distribution flows
    ("Normal MQTT (active TCP)", "In-Dist", "Normal", make_flow(**{
        "mqtt.hdrflags": 48.0, "mqtt.ver": 4.0, "mqtt.len": 39.0,
        "mqtt.topic_len": 24.0, "mqtt.protoname": "MQTT",
        "mqtt.topic": "Temperature_and_Humidity", "mqtt_msg_len": 15.0,
        "tcp.flags": 24.0, "tcp.flags.ack": 1.0, "tcp.ack": 145.0,
        "tcp.checksum": 31042.0, "tcp.len": 41.0, "tcp_dstport_tier": 2
    })),
    ("ICMP Ping Flood", "In-Dist", "DDoS_ICMP", make_flow(**{
        "icmp.checksum": 41205.0, "icmp.seq_le": 18432.0
    })),
    ("TCP Port Scan (SYN/RST)", "In-Dist", "Port_Scanning", make_flow(**{
        "tcp.connection.syn": 1.0, "tcp.connection.rst": 1.0,
        "tcp.flags": 20.0, "tcp_dstport_tier": 1
    })),
    ("TCP SYN Flood", "In-Dist", "DDoS_TCP", make_flow(**{
        "tcp.connection.syn": 1.0, "tcp.flags": 2.0,
        "tcp.checksum": 52190.0, "tcp_options_len": 24.0, "tcp_dstport_tier": 1
    })),
    ("Web Injection (Long Query)", "In-Dist", "SQL_injection / XSS", make_flow(**{
        "http.request.method": "GET", "http.request.version": "HTTP/1.1",
        "http.content_length": 280.0, "http_request_full_uri_len": 145.0,
        "http_request_uri_query_len": 85.0, "tcp.flags.ack": 1.0,
        "tcp_payload_len": 320.0, "tcp_dstport_tier": 1
    })),
    ("Malicious File Upload", "In-Dist", "Uploading", make_flow(**{
        "http.request.method": "POST", "http.request.version": "HTTP/1.1",
        "http.content_length": 1450.0, "http_request_full_uri_len": 64.0,
        "http_file_data_len": 1200.0, "tcp.flags": 24.0, "tcp.flags.ack": 1.0,
        "tcp.len": 1448.0, "tcp_payload_len": 2896.0, "tcp_dstport_tier": 1
    })),
    # Pipeline resilience
    ("Malformed Stream (NaN + PATCH)", "Resilience", "No Crash", make_flow(**{
        "http.request.method": "PATCH", "http.request.version": "HTTP/2.0",
        "http.content_length": np.nan, "tcp.ack": np.nan,
        "tcp.flags.ack": 1.0, "tcp_dstport_tier": 1
    })),
    ("Unseen Campus MQTT Topic", "Resilience", "Normal", make_flow(**{
        "mqtt.hdrflags": 48.0, "mqtt.ver": 4.0, "mqtt.len": 50.0,
        "mqtt.topic_len": 28.0, "mqtt.protoname": "MQTT",
        "mqtt.topic": "Campus_Library_CO2_Sensor_04", "mqtt_msg_len": 22.0,
        "tcp.flags": 24.0, "tcp.flags.ack": 1.0, "tcp.ack": 210.0,
        "tcp.checksum": 44120.0, "tcp.len": 52.0, "tcp_dstport_tier": 2
    })),
    # Out-of-distribution campus traffic
    ("Zero-Day DNS Tunneling", "OOD", "Anomaly", make_flow(**{
        "dns.qry.name": 9999.0, "dns.qry.name.len": "a8f92c.exfil.evil-c2.net",
        "dns.qry.qu": 1.0, "udp_port_tier": 1, "udp.time_delta": 0.002
    })),
    ("Campus Web Portal POST", "OOD", "Normal (FPR check)", make_flow(**{
        "http.request.method": "POST", "http.request.version": "HTTP/1.1",
        "http.content_length": 512.0, "http_request_full_uri_len": 48.0,
        "tcp.flags.ack": 1.0, "tcp_dstport_tier": 1
    })),
    ("Campus UDP Video Stream", "OOD", "Normal (FPR check)", make_flow(**{
        "udp_port_tier": 3, "udp.time_delta": 0.033
    })),
    ("Normal HTTPS TLS Handshake", "OOD", "Normal (FPR check)", make_flow(**{
        "tcp.connection.synack": 1.0, "tcp.flags": 18.0, "tcp.flags.ack": 1.0,
        "tcp.ack": 1.0, "tcp.checksum": 28761.0, "tcp_options_len": 24.0,
        "tcp_dstport_tier": 1
    })),
]

X_stress = pd.DataFrame([s[3] for s in scenarios])[X_train.columns]
probs = xgb_eng.predict_proba(preprocessor.transform(X_stress))
preds = le.inverse_transform(np.argmax(probs, axis=1))
confs = np.max(probs, axis=1) * 100
entropies = -(probs * np.log2(probs + 1e-12)).sum(axis=1)

gate_verdicts = [
    "Unknown Anomaly" if (c < 50.0 or e > 2.5) else ("Normal" if p == "Normal" else f"Attack ({p})")
    for p, c, e in zip(preds, confs, entropies)
]

display(pd.DataFrame({
    "Scenario": [s[0] for s in scenarios],
    "Group": [s[1] for s in scenarios],
    "Expected": [s[2] for s in scenarios],
    "Raw Pred": preds,
    "Conf (%)": np.round(confs, 2),
    "Entropy": np.round(entropies, 3),
    "Gate Verdict": gate_verdicts,
}))

## 5. Export Engineered Dataset & Model Pipeline

Save the 40-feature dataset to Parquet and bundle the fitted `preprocessor` + `xgb_eng` into a single Scikit-Learn `Pipeline` for FastAPI inference.

In [ ]:
import os
import json
import joblib

os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)

df.to_parquet("data/processed/edge_iiot_engineered.parquet", index=False)
df.to_csv("data/processed/edge_iiot_engineered.csv", index=False)

ids_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb_eng),
])
joblib.dump(ids_pipeline, "models/ids_pipeline.joblib", compress=3)
joblib.dump(le, "models/label_encoder.joblib", compress=3)

metrics = {
    "model": "XGBoost",
    "num_features": int(X_train_proc.shape[1]),
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "classes": le.classes_.tolist(),
    "multiclass_accuracy": round(float(accuracy_score(y_test, y_pred_xgb)), 4),
    "macro_f1": round(float(f1_score(y_test, y_pred_xgb, average="macro")), 4),
    "binary_accuracy": round(float(accuracy_score(y_test_bin, y_pred_bin)), 4),
    "binary_f1": round(float(f1_score(y_test_bin, y_pred_bin)), 4),
    "soc_gate": {"min_confidence_pct": 50.0, "max_entropy_bits": 2.5},
}
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Verify end-to-end reload on raw X_test
reloaded = joblib.load("models/ids_pipeline.joblib")
assert np.array_equal(reloaded.predict(X_test.iloc[:10]), y_pred_xgb[:10])
print("Exported:", sorted(os.listdir("models")))

## Summary & Observed Limitations

- **Holdout Results:** Adding 8 structural features (6 payload/URI lengths + 2 port tiers) lifted 15-class XGBoost from 86.05% to **90.86% accuracy (0.9119 Macro F1)** and LightGBM from 85.91% to **90.80% (0.9112 Macro F1)**. Derived binary detection reached **99.68% accuracy (0.9980 F1)** with 2 false alarms out of 4,820 normal test flows.
- **Production Gaps Observed in Stress Testing:**
  1. Single-packet inspection cannot catch volumetric TCP SYN floods without rolling flow-rate features.
  2. Edge-IIoTset `Normal` traffic has no active HTTP bodies (`http.content_length > 0`) or campus UDP streams, causing domain-shift false alarms on benign web POSTs and UDP video traffic.
- **Next Steps:** Move this feature engineering and training logic into modular `src/` scripts with `dvc.yaml`, benchmark UNSW-NB15 as the secondary baseline, and wrap `models/ids_pipeline.joblib` + the confidence/entropy gate in FastAPI + Docker. SHAP explainability and campus baseline fine-tuning are scheduled for Semester 2.